In [4]:
import cv2
import mediapipe as mp
import math
import numpy as np
import os
import urllib.request

MODEL_PATH = 'hand_landmarker.task'
MODEL_URL = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task'

if not os.path.exists(MODEL_PATH):
    print("Descargando modelo...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("Modelo descargado.")
else:
    print("Modelo listo.")

BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=1,
    min_hand_detection_confidence=0.7,
    min_hand_presence_confidence=0.6,
    min_tracking_confidence=0.6
)

detector = HandLandmarker.create_from_options(options)
print("Detector de manos listo.")

Modelo listo.
Detector de manos listo.


In [5]:
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20)
]

def dedo_levantado(landmarks, dedo):
    if dedo == 'pulgar':
        return landmarks[4].x < landmarks[3].x
    elif dedo == 'indice':
        return landmarks[8].y < landmarks[6].y
    elif dedo == 'medio':
        return landmarks[12].y < landmarks[10].y
    elif dedo == 'anular':
        return landmarks[16].y < landmarks[14].y
    elif dedo == 'meñique':
        return landmarks[20].y < landmarks[18].y
    return False

def dedos_levantados(landmarks):
    dedos = ['pulgar', 'indice', 'medio', 'anular', 'meñique']
    return [d for d in dedos if dedo_levantado(landmarks, d)]

def calcular_distancia(p1, p2):
    return math.hypot(p2[0] - p1[0], p2[1] - p1[1])

def dibujar_ui(frame, porcentaje, distancia, dist_max, levantados, altura, ancho):
    C_BG = (18, 18, 22)
    C_ACCENT = (0, 210, 140)
    C_WARN = (0, 180, 255)
    C_LOW = (60, 80, 220)
    C_TEXT = (220, 220, 225)
    C_MUTED = (100, 105, 115)
    C_BORDER = (45, 48, 58)

    panel_x, panel_y = 12, 12
    panel_w, panel_h = 120, altura - 24
    overlay = frame.copy()
    cv2.rectangle(overlay, (panel_x, panel_y),
                  (panel_x + panel_w, panel_y + panel_h), C_BG, cv2.FILLED)
    frame = cv2.addWeighted(overlay, 0.75, frame, 0.25, 0)
    cv2.rectangle(frame, (panel_x, panel_y),
                  (panel_x + panel_w, panel_y + panel_h), C_BORDER, 1)

    bx = panel_x + 20
    by = panel_y + 55
    bw = 22
    bh = panel_h - 110

    cv2.rectangle(frame, (bx, by), (bx + bw, by + bh), C_BORDER, cv2.FILLED)
    cv2.rectangle(frame, (bx, by), (bx + bw, by + bh), C_MUTED, 1)

    fill_h = int(bh * porcentaje / 100)
    fy = by + bh - fill_h
    if porcentaje > 65:
        color_barra = C_ACCENT
    elif porcentaje > 30:
        color_barra = C_WARN
    else:
        color_barra = C_LOW
    if fill_h > 0:
        cv2.rectangle(frame, (bx, fy), (bx + bw, by + bh), color_barra, cv2.FILLED)

    for pct in [0, 50, 100]:
        mark_y = by + bh - int(bh * pct / 100)
        cv2.line(frame, (bx + bw + 2, mark_y), (bx + bw + 8, mark_y), C_MUTED, 1)
        cv2.putText(frame, f"{pct}", (bx + bw + 10, mark_y + 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.32, C_MUTED, 1)

    pct_label = f"{int(porcentaje)}%"
    cv2.putText(frame, pct_label, (panel_x + 8, by + bh + 28),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, color_barra, 2)

    cv2.putText(frame, "CTRL", (panel_x + 28, panel_y + 22),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, C_MUTED, 1)
    cv2.line(frame, (panel_x + 8, panel_y + 28),
             (panel_x + panel_w - 8, panel_y + 28), C_BORDER, 1)

    hud_x = ancho - 260
    hud_y = 12
    hud_w = 248
    hud_h = 90
    overlay2 = frame.copy()
    cv2.rectangle(overlay2, (hud_x, hud_y),
                  (hud_x + hud_w, hud_y + hud_h), C_BG, cv2.FILLED)
    frame = cv2.addWeighted(overlay2, 0.75, frame, 0.25, 0)
    cv2.rectangle(frame, (hud_x, hud_y),
                  (hud_x + hud_w, hud_y + hud_h), C_BORDER, 1)

    cv2.putText(frame, "AIR CONTROLLER", (hud_x + 10, hud_y + 22),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, C_ACCENT, 2)
    cv2.putText(frame, "MediaPipe Hands", (hud_x + 10, hud_y + 42),
                cv2.FONT_HERSHEY_SIMPLEX, 0.38, C_MUTED, 1)
    cv2.putText(frame, f"dist: {int(distancia)}px  max: {int(dist_max)}px",
                (hud_x + 10, hud_y + 62),
                cv2.FONT_HERSHEY_SIMPLEX, 0.36, C_MUTED, 1)
    cv2.putText(frame, "[Q] salir",
                (hud_x + 10, hud_y + 80),
                cv2.FONT_HERSHEY_SIMPLEX, 0.34, C_MUTED, 1)

    bar_y = altura - 34
    overlay3 = frame.copy()
    cv2.rectangle(overlay3, (0, bar_y), (ancho, altura), C_BG, cv2.FILLED)
    frame = cv2.addWeighted(overlay3, 0.72, frame, 0.28, 0)

    if levantados is not None:
        texto = "dedos: " + (", ".join(levantados) if levantados else "ninguno")
        cv2.putText(frame, texto, (panel_x + panel_w + 16, altura - 12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.48, C_TEXT, 1)
    else:
        cv2.putText(frame, "sin mano detectada", (panel_x + panel_w + 16, altura - 12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.48, C_MUTED, 1)

    return frame

print("Funciones auxiliares definidas.")

Funciones auxiliares definidas.


In [6]:
from collections import deque

DIST_MIN = 20
DIST_MAX = 300

HIST_SIZE = 120
MARGIN_LOW = 0.05
MARGIN_HIGH = 0.95
dist_history = deque(maxlen=HIST_SIZE)

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("No se pudo abrir la cámara.")
else:
    print("Cámara abierta. Presiona Q para salir.")
    print("Abre y cierra los dedos en los primeros segundos para calibrar.")

porcentaje_actual = 0
timestamp = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    altura, ancho, _ = frame.shape

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    timestamp += 33

    results = detector.detect_for_video(mp_image, timestamp)

    levantados = None
    distancia_actual = 0

    if results.hand_landmarks:
        for landmarks in results.hand_landmarks:
            for connection in HAND_CONNECTIONS:
                pt1 = landmarks[connection[0]]
                pt2 = landmarks[connection[1]]
                x1, y1 = int(pt1.x * ancho), int(pt1.y * altura)
                x2, y2 = int(pt2.x * ancho), int(pt2.y * altura)
                cv2.line(frame, (x1, y1), (x2, y2), (55, 60, 70), 1)

            for lm in landmarks:
                x, y = int(lm.x * ancho), int(lm.y * altura)
                cv2.circle(frame, (x, y), 3, (80, 200, 140), cv2.FILLED)

            lm = landmarks
            pulgar_x = int(lm[4].x * ancho)
            pulgar_y = int(lm[4].y * altura)
            indice_x = int(lm[8].x * ancho)
            indice_y = int(lm[8].y * altura)

            distancia_actual = calcular_distancia(
                (pulgar_x, pulgar_y), (indice_x, indice_y))

            dist_history.append(distancia_actual)
            if len(dist_history) >= 20:
                sorted_hist = sorted(dist_history)
                n = len(sorted_hist)
                DIST_MIN = max(10, sorted_hist[int(n * MARGIN_LOW)])
                DIST_MAX = max(DIST_MIN + 30,
                               sorted_hist[int(n * MARGIN_HIGH)])

            if distancia_actual >= DIST_MAX:
                porcentaje = 100.0
            else:
                porcentaje = (distancia_actual - DIST_MIN) / (DIST_MAX - DIST_MIN) * 100
                porcentaje = float(np.clip(porcentaje, 0, 100))
            
            porcentaje_actual = porcentaje_actual * 0.7 + porcentaje * 0.3

            if porcentaje_actual < 20:
                col = (60, 80, 220)
            elif porcentaje_actual < 65:
                col = (0, 180, 255)
            else:
                col = (0, 210, 140)

            cv2.circle(frame, (pulgar_x, pulgar_y), 13, (255, 160, 30), cv2.FILLED)
            cv2.circle(frame, (pulgar_x, pulgar_y), 13, (255, 255, 255), 1)
            cv2.circle(frame, (indice_x, indice_y), 13, (30, 160, 255), cv2.FILLED)
            cv2.circle(frame, (indice_x, indice_y), 13, (255, 255, 255), 1)
            cv2.line(frame, (pulgar_x, pulgar_y),
                     (indice_x, indice_y), col, 2)

            cx = (pulgar_x + indice_x) // 2
            cy = (pulgar_y + indice_y) // 2
            cv2.circle(frame, (cx, cy), 6, col, cv2.FILLED)

            levantados = dedos_levantados(lm)

    frame = dibujar_ui(
        frame, porcentaje_actual, distancia_actual,
        DIST_MAX, levantados, altura, ancho
    )

    cv2.imshow("Air Controller — MediaPipe Hands", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("Saliendo...")
        break

cap.release()
cv2.destroyAllWindows()
print("Recursos liberados correctamente.")

Cámara abierta. Presiona Q para salir.
Abre y cierra los dedos en los primeros segundos para calibrar.
Saliendo...
Recursos liberados correctamente.
